In [29]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [30]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [31]:
# Time
start = dt.datetime(2019,4,5)
end = dt.datetime(2019,5,13)
print(start,end,end-start)

2019-04-05 00:00:00 2019-05-13 00:00:00 38 days, 0:00:00


In [32]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df = df[df["sign_up_details_app_platform"] == "UNITY_Android"]
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
len(users)
print(users.head())

                    user_id              createtime  \
0  5ca6adb3b65b15544e169963 2019-04-05 01:21:55.931   
1  5ca6c1bc8c899454486ac253 2019-04-05 02:47:24.829   
2  5ca6d01fd468e03534a80f6d 2019-04-05 03:48:47.021   
4  5ca6e3fc800ebb353a469cd1 2019-04-05 05:13:32.630   
5  5ca6e9a3acb9c30617a1e75f 2019-04-05 05:37:39.265   

                          device_id            last_request  
0  ba643254ecf37ea8a8453a12ab14e7a8 2019-04-05 01:22:16.059  
1  3ac01ce3f63c2c857064702d99c80541 2019-04-05 03:09:53.215  
2  2534dd971bd5601f2985c94d1da74cc9 2019-04-05 03:51:56.000  
4  bdfe19ca2338a8176670c5c79695b36f 2019-04-05 10:11:34.754  
5  a395477c3bd6830b7e0c68571da67ed8 2019-04-05 05:37:44.384  


In [35]:
c_payment = cursor.superstars.payment_orders
aw = []
for documents in c_payment.find({'status':2,'created_at':{'$gte': start}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df = df.rename(columns={'revenue_top_line':'money'})
df = df[(df["user_name"] != "FirzenYogesh")&(df["user_name"] != "MrYoBear")&(df["user_name"] != "goyaala")&(df['money']>0)]
df.sort_values('money',ascending=False,inplace=True)
print(df.head())

    __v                       _id              created_at currency_code  \
70    0  5ccb793df2bcf7711bef1020 2019-05-02 23:11:57.906           INR   
92    0  5cd96a9c26267106967efc1b 2019-05-13 13:01:16.405           INR   
42    0  5cbe10af9cde200800879b9e 2019-04-22 19:06:23.524           NaN   
90    0  5cd89eaf62ebdd1063851e0f 2019-05-12 22:31:11.573           INR   
25    0  5cb2a53fa47e2d78d2a59997 2019-04-14 03:13:03.925           NaN   

          details_gateway details_misc_bankcode details_misc_gatewayname  \
70  google_in_app_billing                   NaN                      NaN   
92  google_in_app_billing                   NaN                      NaN   
42  google_in_app_billing                   NaN                      NaN   
90  google_in_app_billing                   NaN                      NaN   
25  google_in_app_billing                   NaN                      NaN   

   details_misc_payment_method          details_order_id  \
70                         NaN  

In [39]:
f = {'money':'sum','user_name':'first'}
df1 = df.groupby("user_id").agg(f)
#df1 = df1.to_frame().reset_index()
df1.sort_values('money',ascending = False,inplace = True)
print(df1.head())
Total = df['money'].sum()
print (Total/len(df1))
print(len(df1))

                           money   user_name
user_id                                     
5cc4a02cd3d9160b131fa69a  7900.0  Flintstone
5ca1a83dc0a01e6c61ee39d1  2098.0    Jon Snow
5cabff951f395e1f8330c4c1  1599.0       Vijay
5cd696d2c513ef678e74c36d  1599.0     Optimus
5ca988ad6504284e4ce6925a  1599.0       subha
550.9130434782609
46


In [36]:
f = {'user_id':'count','money':'sum'}
df2 = df.groupby(['items_0_id',"items_0_type"]).agg(f)
print(df2)

                                  user_id   money
items_0_id items_0_type                          
1          HARD_CURRENCY_PACKAGE       26  1274.0
2          HARD_CURRENCY_PACKAGE       14  2786.0
3          HARD_CURRENCY_PACKAGE       14  6986.0
4          HARD_CURRENCY_PACKAGE        4  6396.0
6          HARD_CURRENCY_PACKAGE        1  7900.0
